# LLM-as-a-Judge, AlpacaEval 2.0

Единый ноутбук: отбор задач, генерация ответов/вариантов ответов локальными моделями и прогон судей.
Рассчитан на Kaggle (GPU), **устойчив к обрыву сессии**: все результаты записываются в csv в папку `data/` рядом с ноутбуком после каждого батча, любая ячейка запуска пропускает уже сделанное.

**Пайплайн** (модели работают по очереди, после того, как одна завершает работу, происходит выгрузка из cuda):
1. `Stage 1` - оцениваемые модели (SmolLM2, Gemma-2) отвечают на свои задачи (вариант **A**).
2. `Stage 2` - Phi-3.5-mini строит варианты **B, C, D, E, P** из ответа A.
3. `Stage 3` - судьи (Qwen, Mistral-Nemo, Llama) выносят вердикты по запланированным парам.

**Как продолжить после остановки:** откройте ноутбук, выполните ячейки с настройками/функциями и `Init`, затем снова `Stage 1 → 2 → 3` (или `run_all()`). Если файлов `data/*.csv` в папке ноутбука нет, а результаты прошлого запуска подключены как Kaggle Input - они будут восстановлены автоматически.


In [1]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.9 MB/s eta 0:00:00


In [1]:
import os, gc, re, json, time, shutil, hashlib
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

_T0 = time.time()

# ============================== КОНФИГУРАЦИЯ ==============================
SEED = 42
N_TASKS = 250                  # 200-300
ACTIVE_TASK_LIMIT = 5 #None       # None = все задачи; число K = тест на первых K задачах

# Файлы создаются\читаются в директории ноутбука
DATA_DIR = Path.cwd() / "data"
TASKS_CSV       = DATA_DIR / "tasks.csv"
VARIANTS_CSV    = DATA_DIR / "variants.csv"
COMPARISONS_CSV = DATA_DIR / "comparisons.csv"
JUDGMENTS_CSV   = DATA_DIR / "judgments.csv"
ATTEMPTS_CSV    = DATA_DIR / "answer_attempts.csv"   # журнал попыток генерации ответов A

# Модели
EVALUATED_MODELS = [                       # каждая отвечает на все задачи, создается вариант A для каждой задачи моделью
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "google/gemma-2-9b-it",
]
SOURCE_TAGS = {                            # короткие метки в variant_id/comparison_id 
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": "smol",
    "google/gemma-2-9b-it": "gemma",
}
REWRITER_MODEL = "microsoft/Phi-3.5-mini-instruct"    # вариант P
GENERATOR_MODEL = REWRITER_MODEL                      # варианты B, C, D, E
JUDGE_MODELS = {                                      # метка (в comparisons.csv)
    "judge_qwen":    "Qwen/Qwen2.5-7B-Instruct",
    "judge_mistral": "mistralai/Mistral-Nemo-Instruct-2407",
    "judge_llama":   "meta-llama/Llama-3.2-3B-Instruct",
}
JUDGES = list(JUDGE_MODELS)
PRIVATE_MODELS = {"meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-9b-it"}  # нужен HF_TOKEN

# Уровни длины
EXPAND_LEVELS   = [1.25, 1.50, 2.00]       # B и C
COMPRESS_LEVELS = [0.75, 0.50]             # D
LOSSY_LEVELS    = [0.75]                   # E
REPEATS = 2                                # repeat_index 0/1: позиции left/right меняются местами

# Генерация
USE_4BIT = True
BATCH_GEN = 4                              # батч при генерации ответов и вариантов
BATCH_JUDGE = 8                            # батч при судействе
A_MAX_NEW_TOKENS = 700
JUDGE_MAX_NEW_TOKENS = 8
TIME_BUDGET_HOURS = 11.0                   # мягкая остановка до лимита сессии Kaggle (ч)

A_MIN_WORDS, A_MAX_WORDS = 100, 450        # допустимая длина ответа: от неё зависит возможность сжатия до 50% и расширения до 200%
A_MAX_REGEN = 3                            # максимум перегенераций одного ответа
A_TARGET_WORDS = 275                       # ориентир длины в подсказке при перегенерации

# Критерии отбора (по ответу AlpacaEval)
MIN_WORDS, MAX_WORDS = 100, 450
MIN_SENTENCES = 6
MAX_CODE_FRACTION = 0.30
MIN_ASCII_RATIO = 0.95

TASK_COLS = ["task_id", "source_benchmark", "benchmark_id", "instruction",
             "answers", "gold_answer", "category", "selection_reason"]
VARIANT_COLS = ["variant_id", "task_id", "variant_type", "text", "target_length_ratio",
                "actual_length_ratio", "generator_model", "prompt"]
COMPARISON_COLS = ["comparison_id", "task_id", "experiment_type", "variant_left_id",
                   "variant_right_id", "judge_id", "repeat_index", "random_seed"]
JUDGMENT_COLS = ["judgment_id", "comparison_id", "winner", "raw_response",
                 "latency_ms", "input_tokens", "output_tokens"]
ATTEMPT_COLS = ["variant_id", "attempt", "n_words", "n_tokens", "truncated", "status", "kept", "hint"]

DERIVED_PLAN = ([("B", r) for r in EXPAND_LEVELS] + [("C", r) for r in EXPAND_LEVELS]
                + [("D", r) for r in COMPRESS_LEVELS] + [("E", r) for r in LOSSY_LEVELS]
                + [("P", 1.0)])

PROMPTS = {
    "B": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so it is about "
          "{words} words long. Add ONLY new, relevant, factually correct atomic claims "
          "(extra facts, examples, caveats). Keep every original claim. Output only the answer."),
    "C": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so it is about "
          "{words} words long WITHOUT adding any new facts or claims: only paraphrase, add "
          "explanatory connectives, restate and structure. Output only the answer."),
    "D": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nCondense the answer to about "
          "{words} words while keeping ALL essential atomic claims. Remove only redundancy. "
          "The text must stay fluent. Output only the answer."),
    "E": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nShorten the answer to about "
          "{words} words by DELETING some essential points (about a third of the key claims). "
          "Do not add anything new. Output only the answer."),
    "P": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer in different "
          "wording and style with the SAME length (~{words} words) and exactly the same "
          "claims. Output only the answer."),
}

JUDGE_PROMPT = ("You are an impartial judge. Below are an instruction and two responses to it. "
                "Decide which response is better. Do not let the order of the responses influence "
                "your decision.\n\n[Instruction]\n{instruction}\n\n[Response A]\n{left}\n\n"
                "[Response B]\n{right}\n\nWhich response is better? "
                "Answer with a single letter: A or B.")


In [21]:
# ============================== HF TOKEN ==============================
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("ВНИМАНИЕ: HF_TOKEN не найден - gated-модели (Llama, Gemma) не загрузятся. "
          "Добавьте токен в Kaggle Secrets.")

quantization_config = None
if torch.cuda.is_available() and USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

_loaded_models = {}


def get_hf_auth_kwargs(model_id):
    return {"token": HF_TOKEN} if (model_id in PRIVATE_MODELS and HF_TOKEN) else {}


def gpu_mem():
    if not torch.cuda.is_available():
        return "cpu"
    return ", ".join(f"cuda:{i} {torch.cuda.memory_allocated(i)/2**30:.1f} GiB"
                     for i in range(torch.cuda.device_count()))


def load_local_model(model_id):
    if model_id in _loaded_models:
        return _loaded_models[model_id]
    auth = get_hf_auth_kwargs(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left", **auth)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"device_map": "auto",
              "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32, **auth}
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    _loaded_models[model_id] = (tokenizer, model)
    print(f"[load] {model_id} | GPU: {gpu_mem()}")
    return _loaded_models[model_id]


def unload_model(model_id):
    """Полностью освобождает CUDA под следующую модель."""
    entry = _loaded_models.pop(model_id, None)
    if entry is not None:
        tokenizer, model = entry
        del entry, tokenizer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print(f"[unload] {model_id} | GPU: {gpu_mem()}")


@torch.inference_mode()
def generate_batch(model_id, prompts, max_new_tokens, seed=None, **gen_kwargs):
    """Жадная генерация батча. Возвращает список dict(text, input_tokens, output_tokens, latency_ms).
    latency_ms - время батча/размер батча."""
    tokenizer, model = load_local_model(model_id)
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True)
             for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)   # BOS уже есть в chat-шаблоне
    t0 = time.perf_counter()
    if seed is not None:
        torch.manual_seed(seed)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                         **{"do_sample": False, **gen_kwargs})
    elapsed_ms = (time.perf_counter() - t0) * 1000
    in_len = enc["input_ids"].shape[1]
    in_tokens = enc["attention_mask"].sum(dim=1).tolist()
    res = []
    for i in range(len(prompts)):
        new = out[i, in_len:]
        res.append({"text": tokenizer.decode(new, skip_special_tokens=True).strip(),
                    "input_tokens": int(in_tokens[i]),
                    "output_tokens": int((new != tokenizer.pad_token_id).sum()),
                    "latency_ms": elapsed_ms / len(prompts)})
    return res


def generate_safe(model_id, prompts, max_new_tokens):
    """При нехватке памяти батч делится пополам."""
    try:
        return generate_batch(model_id, prompts, max_new_tokens, **gen_kwargs)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        mid = len(prompts) // 2
        return (generate_safe(model_id, prompts[:mid], max_new_tokens, **gen_kwargs)
                + generate_safe(model_id, prompts[mid:], max_new_tokens, **gen_kwargs))


In [15]:
# ============================== Вспомогательные функции ==============================
def n_words(text): return len(text.split())
def n_sentences(text): return len([s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.split()) >= 3])
def code_fraction(text):
    code = sum(len(m) for m in re.findall(r"```.*?```", text, flags=re.S))
    return code / max(len(text), 1)
def ascii_ratio(text): return sum(c.isascii() for c in text) / max(len(text), 1)
def stable_int(*parts, mod=2**31 - 1):
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest(), 16) % mod
def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def variant_id(task_id, src, vt, ratio=1.0):
    """T0001_smol_A, T0001_smol_B125, ... src - метка оцениваемой модели, чей ответ A расширяется/сжимается."""
    return f"{task_id}_{src}_A" if vt == "A" else f"{task_id}_{src}_{vt}{round(ratio * 100)}"

def a_id_of(vid):
    t, s = vid.split("_")[:2]
    return f"{t}_{s}_A"

# ============================== CSV вывод и запись ==============================
def read_csv(path, cols):
    df = pd.read_csv(path, dtype=object, keep_default_na=False)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: нет колонок {missing}")
    return df[cols].copy()

def write_csv_atomic(df, path):
    """Запись через временный файл"""
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        df.to_csv(f, index=False)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# ============================== 1. tasks ==============================
def load_alpaca_eval():
    url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
    df = pd.read_json(url).reset_index(drop=True)      
    df.insert(0, "benchmark_id", range(len(df)))
    return df                 # benchmark_id, instruction, output, generator, dataset

def select_tasks(df, n_tasks):
    df = df.copy()
    df["words"] = df["output"].map(n_words)
    df["sents"] = df["output"].map(n_sentences)
    df["code_frac"] = df["output"].map(code_fraction)
    df["ascii"] = (df["instruction"] + df["output"]).map(ascii_ratio)
    ok = df[(df.words.between(MIN_WORDS, MAX_WORDS)) & (df.sents >= MIN_SENTENCES)
            & (df.code_frac <= MAX_CODE_FRACTION) & (df.ascii >= MIN_ASCII_RATIO)].copy()
    ok = ok.drop_duplicates("instruction")
    if len(ok) < 200:
        raise RuntimeError(f"После фильтрации осталось {len(ok)} задач (<200): ослабьте критерии.")
    n_tasks = min(n_tasks, len(ok))
    shares = ok["dataset"].value_counts(normalize=True)          # стратификация по подмножествам
    parts = []
    for cat, share in shares.items():
        sub = ok[ok.dataset == cat]
        parts.append(sub.sample(n=min(max(1, round(share * n_tasks)), len(sub)), random_state=SEED))
    sel = pd.concat(parts)
    if len(sel) > n_tasks:
        sel = sel.sample(n=n_tasks, random_state=SEED)
    elif len(sel) < n_tasks:
        sel = pd.concat([sel, ok.drop(sel.index).sample(n=n_tasks - len(sel), random_state=SEED)])
    sel = sel.sort_values("benchmark_id").reset_index(drop=True)
    rows = []
    for i, r in sel.iterrows():
        rows.append({
            "task_id": f"T{i:04d}", "source_benchmark": "AlpacaEval 2.0",
            "benchmark_id": int(r.benchmark_id), "instruction": r.instruction,
            "answers": "[]",                       # заполняется после Stage 1 (ответ оцениваемой модели)
            "gold_answer": r.output,               # референс AlpacaEval (GPT-4 Turbo), не строгий gold
            "category": r.dataset,
            "selection_reason": (f"reference {r.words} words, {r.sents} sentences, "
                                 f"code_frac={r.code_frac:.2f}: enough content for 50-75% compression "
                                 f"and 125-200% expansion; stratified by subset '{r.dataset}'")})
    return pd.DataFrame(rows, columns=TASK_COLS)


# ============================== 2. variants ==============================
def ensure_a_rows(variants, tasks):
    """Строки A для всех задач; каждая модель отвечает на каждую задачу."""
    have = set(variants.variant_id)
    new = []
    '''
    for i, t in enumerate(tasks.itertuples()):
        vid = variant_id(t.task_id, "A")
        if vid not in have:
            new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                        "target_length_ratio": 1.0, "actual_length_ratio": "",
                        #"generator_model": EVALUATED_MODELS[i % len(EVALUATED_MODELS)],
                        "generator_model": EVALUATED_MODELS[i],
                        "prompt": t.instruction})
    '''
    for model in EVALUATED_MODELS:
        for t in tasks.itertuples():
            vid = variant_id(t.task_id, SOURCE_TAGS[model], "A")
            if vid not in have:
                new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                            "target_length_ratio": 1.0, "actual_length_ratio": "",
                            "generator_model": model, "prompt": t.instruction})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants

def ensure_derived_rows(variants, tasks):
    """Строки B/C/D/E/P с готовым prompt - только для задач, у которых уже есть текст A."""
    have = set(variants.variant_id)
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    instr = dict(zip(tasks.task_id, tasks.instruction))
    new = []
    for r in a.itertuples():
        src = SOURCE_TAGS[r.generator_model]
        a_words = n_words(r.text)
        for vt, ratio in DERIVED_PLAN:
            vid = variant_id(r.task_id, src, vt, ratio)
            if vid in have:
                continue
            prompt = PROMPTS[vt].format(instruction=instr[r.task_id], answer=r.text,
                                        words=round(a_words * ratio))
            new.append({"variant_id": vid, "task_id": r.task_id, "variant_type": vt, "text": "",
                        "target_length_ratio": ratio, "actual_length_ratio": "",
                        "generator_model": REWRITER_MODEL if vt == "P" else GENERATOR_MODEL,
                        "prompt": prompt})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants


# ============================== 3. comparisons ==============================
def build_comparisons(task_ids):
    rows = []
    def add(task_id, src, exp, v1, v2):
        for judge in JUDGES:
            for rep in range(REPEATS):
                left, right = (v1, v2) if rep % 2 == 0 else (v2, v1)
                cid = f"{task_id}_{src}_{exp}_{v1.split('_')[2]}_{v2.split('_')[2]}_{judge}_r{rep}"
                rows.append({"comparison_id": cid, "task_id": task_id, "experiment_type": exp,
                             "variant_left_id": left, "variant_right_id": right,
                             "judge_id": judge, "repeat_index": rep,
                             "random_seed": stable_int(cid, SEED)})
    for tid in task_ids:
        for model in EVALUATED_MODELS:                 # сравнения только внутри одного источника A
            src = SOURCE_TAGS[model]
            V = lambda vt, r=1.0: variant_id(tid, src, vt, r)
            for r in EXPAND_LEVELS:
                add(tid, src, "exp1_B_vs_C", V("B", r), V("C", r))
                add(tid, src, "exp2_C_vs_A", V("C", r), V("A"))
            for r in COMPRESS_LEVELS:
                add(tid, src, "exp3_D_vs_A", V("D", r), V("A"))
            for r in LOSSY_LEVELS:
                add(tid, src, "exp4_E_vs_A", V("E", r), V("A"))
            add(tid, src, "exp5_P_vs_A", V("P"), V("A"))
    return pd.DataFrame(rows, columns=COMPARISON_COLS)


In [16]:
# =============== Init: файлы создаются, если их нет; если есть - загрузка и продолжение ===============
DATA_DIR.mkdir(parents=True, exist_ok=True)

def restore_from_previous_run():
    """Если в папке ноутбука данных нет, а результаты прошлого запуска подключены как Kaggle Input — копируем их."""
    if TASKS_CSV.exists():
        return
    root = Path("/kaggle/input")
    if not root.exists():
        return
    names = ["tasks.csv", "variants.csv", "comparisons.csv", "judgments.csv"]
    for p in root.rglob("tasks.csv"):
        if all((p.parent / n).exists() for n in names):
            for n in names:
                shutil.copy(p.parent / n, DATA_DIR / n)
            print(f"[restore] данные восстановлены из {p.parent}")
            return

restore_from_previous_run()

# tasks
if TASKS_CSV.exists():
    tasks = read_csv(TASKS_CSV, TASK_COLS); print(f"[tasks] загружен: {len(tasks)} задач")
else:
    tasks = select_tasks(load_alpaca_eval(), N_TASKS)
    write_csv_atomic(tasks, TASKS_CSV); print(f"[tasks] создан: {len(tasks)} задач")

# variants (строки A)
if VARIANTS_CSV.exists():
    variants = read_csv(VARIANTS_CSV, VARIANT_COLS); print(f"[variants] загружен: {len(variants)} строк")
else:
    variants = pd.DataFrame(columns=VARIANT_COLS, dtype=object); print("[variants] создан")
variants = ensure_a_rows(variants, tasks)
write_csv_atomic(variants, VARIANTS_CSV)

# comparisons (достраивание недостающих id)
plan = build_comparisons(tasks.task_id)
if COMPARISONS_CSV.exists():
    comparisons = read_csv(COMPARISONS_CSV, COMPARISON_COLS)
    comparisons = pd.concat([comparisons, plan[~plan.comparison_id.isin(set(comparisons.comparison_id))]],
                            ignore_index=True)
    print(f"[comparisons] загружен: {len(comparisons)} строк")
else:
    comparisons = plan; print(f"[comparisons] создан: {len(comparisons)} строк")
write_csv_atomic(comparisons, COMPARISONS_CSV)

# judgments
if JUDGMENTS_CSV.exists():
    judgments = read_csv(JUDGMENTS_CSV, JUDGMENT_COLS); print(f"[judgments] загружен: {len(judgments)} строк")
else:
    judgments = pd.DataFrame(columns=JUDGMENT_COLS, dtype=object)
    write_csv_atomic(judgments, JUDGMENTS_CSV); print("[judgments] создан (пустой)")

# журнал попыток генерации ответов A
if ATTEMPTS_CSV.exists():
    attempts = read_csv(ATTEMPTS_CSV, ATTEMPT_COLS); print(f"[attempts] загружен: {len(attempts)} строк")
else:
    attempts = pd.DataFrame(columns=ATTEMPT_COLS, dtype=object)
    write_csv_atomic(attempts, ATTEMPTS_CSV); print("[attempts] создан (пустой)")


[tasks] загружен: 208 задач
[variants] загружен: 416 строк
[comparisons] загружен: 24960 строк
[judgments] загружен: 0 строк
[attempts] загружен: 0 строк


In [19]:
class BudgetExceeded(Exception):
    pass

def check_budget():
    if time.time() - _T0 > TIME_BUDGET_HOURS * 3600:
        raise BudgetExceeded

def active_task_ids():
    ids = tasks.task_id.tolist()
    return set(ids[:ACTIVE_TASK_LIMIT]) if ACTIVE_TASK_LIMIT else set(ids)

def sync_task_answers():
    """Переносит ответы A оцениваемых моделей в tasks.answers."""
    global tasks
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    ans = {}
    for r in a.itertuples():
        ans.setdefault(r.task_id, []).append({"generator": r.generator_model, "output": r.text})
    tasks["answers"] = [json.dumps(ans[t], ensure_ascii=False) if t in ans else old
                        for t, old in zip(tasks.task_id, tasks.answers)]
    write_csv_atomic(tasks, TASKS_CSV)


# ---------- генерация (ответы A и варианты B/C/D/E/P) ----------
def run_generation(model_id, unload=True):
    """Дозаполняет все пустые тексты, закреплённые за model_id. После каждого батча сохранение."""
    global variants
    check_budget()
    mask = ((variants.generator_model == model_id) & (variants.text == "")
            & (variants.prompt != "") & variants.task_id.isin(active_task_ids()))
    idx = variants.index[mask].tolist()
    if not idx:
        print(f"[skip] {model_id}: нечего генерировать"); return
    a_words = {r.variant_id: n_words(r.text) for r in variants[variants.variant_type == "A"].itertuples() if r.text}

    def max_new(i):
        row = variants.loc[i]
        if row.variant_type == "A":
            return A_MAX_NEW_TOKENS
        words = round(a_words[a_id_of(row.variant_id)] * float(row.target_length_ratio))
        return min(int(words * 1.5) + 60, 1800)

    idx.sort(key=lambda i: -len(variants.at[i, "prompt"]))     # похожие по длине - в один батч
    print(f"[run] {model_id}: осталось {len(idx)}")
    bar = tqdm(total=len(idx), desc=model_id.split("/")[-1])
    try:
        for chunk in chunks(idx, BATCH_GEN):
            check_budget()
            res = generate_safe(model_id, [variants.at[i, "prompt"] for i in chunk],
                                max(max_new(i) for i in chunk))
            new_log = []
            for i, r in zip(chunk, res):
                if not r["text"]:
                    continue                                   # пустой вывод - повторится при следующем запуске
                variants.at[i, "text"] = r["text"]
                if variants.at[i, "variant_type"] == "A":
                    variants.at[i, "actual_length_ratio"] = 1.0
                    new_log.append(_attempt_row(variants.at[i, "variant_id"], 1, r, "", True))
                else:
                    variants.at[i, "actual_length_ratio"] = round(
                        n_words(r["text"]) / a_words[a_id_of(variants.at[i, "variant_id"])], 3)
            write_csv_atomic(variants, VARIANTS_CSV)
            if new_log:
                log_attempts(new_log)
            bar.update(len(chunk))
    finally:
        bar.close()
        if unload:
            unload_model(model_id)
        sync_task_answers()

        

# ---------- контроль длины ответов A: журнал попыток и перегенерация ----------
import itertools

def answer_status(n_w, truncated=False):
    if truncated:
        return "truncated"
    if n_w > A_MAX_WORDS:
        return "too_long"
    if n_w < A_MIN_WORDS:
        return "too_short"
    return "ok"

def badness(n_w, truncated=False):
    """Насколько ответ вне допустимого диапазона (0 - в диапазоне); обрезанный ответ штрафуется."""
    return max(A_MIN_WORDS - n_w, n_w - A_MAX_WORDS, 0) + (1000 if truncated else 0)

def _attempt_row(vid, attempt, res, hint, kept):
    n_w = n_words(res["text"])
    trunc = int(res["output_tokens"] >= A_MAX_NEW_TOKENS)       # упёрлись в лимит токенов -> ответ обрезан
    return {"variant_id": vid, "attempt": attempt, "n_words": n_w, "n_tokens": res["output_tokens"],
            "truncated": trunc, "status": answer_status(n_w, bool(trunc)), "kept": int(kept), "hint": hint}

def log_attempts(rows):
    global attempts
    attempts = pd.concat([attempts, pd.DataFrame(rows, columns=ATTEMPT_COLS)], ignore_index=True)
    write_csv_atomic(attempts, ATTEMPTS_CSV)

def _attempt_state():
    """{variant_id: (число попыток, обрезан ли сохранённый ответ)}; нет записи в журнале = 1 попытка."""
    if attempts.empty:
        return {}
    a = attempts.copy()
    for c in ("attempt", "truncated", "kept"):
        a[c] = pd.to_numeric(a[c])
    state = {}
    for vid, g in a.groupby("variant_id"):
        kept = g[g.kept == 1]
        state[vid] = (int(g.attempt.max()), bool(kept.truncated.iloc[-1]) if len(kept) else False)
    return state

def answers_needing_regen(model_id):
    """Ответы A модели вне допустимой длины (или обрезанные), у которых остались попытки."""
    state = _attempt_state()
    has_derived = {a_id_of(v) for v in variants.variant_id[variants.variant_type != "A"]}
    a = variants[(variants.variant_type == "A") & (variants.generator_model == model_id)
                 & (variants.text != "") & variants.task_id.isin(active_task_ids())]
    todo, blocked = [], 0
    for i, r in a.iterrows():
        used, trunc = state.get(r.variant_id, (1, False))
        status = answer_status(n_words(r.text), trunc)
        if status == "ok" or used >= 1 + A_MAX_REGEN:
            continue
        if r.variant_id in has_derived:
            blocked += 1
            continue
        todo.append({"idx": i, "vid": r.variant_id, "task_id": r.task_id, "status": status,
                     "attempt": used + 1, "old_bad": badness(n_words(r.text), trunc)})
    if blocked:
        print(f"[regen] пропущено {blocked}: от этих ответов уже построены B-E/P (см. reset_answers)")
    return todo

def regen_prompt(instruction, status):
    if status == "too_short":
        hint = f"Please give a detailed and complete answer of about {A_TARGET_WORDS} words."
    else:                                            # too_long / truncated
        hint = f"Please answer concisely, in about {A_TARGET_WORDS} words."
    return f"{instruction}\n\n{hint}", hint

def regenerate_answers(model_id):
    """Перегенерирует слишком длинные/короткие/обрезанные ответы A (модель остаётся загруженной).
    Попытка 2: жадная генерация + подсказка о длине; попытки 3+: сэмплирование (T=0.7) + подсказка.
    Новый ответ заменяет старый, только если ближе к допустимому диапазону. Использованный промпт пишется в variants.prompt."""
    instr = dict(zip(tasks.task_id, tasks.instruction))
    for _ in range(A_MAX_REGEN):
        todo = answers_needing_regen(model_id)
        if not todo:
            return
        print(f"[regen] {model_id}: {len(todo)} ответов вне [{A_MIN_WORDS}, {A_MAX_WORDS}] слов или обрезаны")
        todo.sort(key=lambda t: t["attempt"])
        for attempt_no, grp in itertools.groupby(todo, key=lambda t: t["attempt"]):
            kw = {} if attempt_no == 2 else {"do_sample": True, "temperature": 0.7, "top_p": 0.9,
                                              "seed": SEED + attempt_no}
            for chunk in chunks(list(grp), BATCH_GEN):
                check_budget()
                built = [regen_prompt(instr[t["task_id"]], t["status"]) for t in chunk]
                res = generate_safe(model_id, [b[0] for b in built], A_MAX_NEW_TOKENS, **kw)
                log = []
                for t, (prompt, hint), r in zip(chunk, built, res):
                    new_bad = (badness(n_words(r["text"]), r["output_tokens"] >= A_MAX_NEW_TOKENS)
                               if r["text"] else float("inf"))
                    keep_new = new_bad < t["old_bad"]
                    if keep_new:
                        variants.at[t["idx"], "text"] = r["text"]
                        variants.at[t["idx"], "prompt"] = prompt
                    log.append(_attempt_row(t["vid"], attempt_no, r, hint, keep_new))
                write_csv_atomic(variants, VARIANTS_CSV)
                log_attempts(log)

def reset_answers(task_ids, model_id=None):
    """Стирает ответы A выбранных задач (по одной или всем оцениваемым моделям) и всё, что из них построено:
    варианты B-E/P, вердикты судей, журнал попыток. После этого Stage 1 сгенерирует эти ответы заново."""
    global variants, judgments, attempts
    tags = [SOURCE_TAGS[m] for m in ([model_id] if model_id else EVALUATED_MODELS)]
    prefixes = tuple(f"{t}_{s}_" for t in task_ids for s in tags)
    chain = variants.variant_id.str.startswith(prefixes)
    chain_ids = set(variants.variant_id[chain])
    instr = dict(zip(tasks.task_id, tasks.instruction))
    comp_ids = set(comparisons.comparison_id[comparisons.variant_left_id.isin(chain_ids)
                                             | comparisons.variant_right_id.isin(chain_ids)])
    variants = variants[~(chain & (variants.variant_type != "A"))].reset_index(drop=True)
    is_a = variants.variant_id.isin(chain_ids) & (variants.variant_type == "A")
    variants.loc[is_a, ["text", "actual_length_ratio"]] = ""
    variants.loc[is_a, "prompt"] = variants.loc[is_a, "task_id"].map(instr)
    judgments = judgments[~judgments.comparison_id.isin(comp_ids)].reset_index(drop=True)
    attempts = attempts[~attempts.variant_id.isin(chain_ids)].reset_index(drop=True)
    for df, path in ((variants, VARIANTS_CSV), (judgments, JUDGMENTS_CSV), (attempts, ATTEMPTS_CSV)):
        write_csv_atomic(df, path)
    sync_task_answers()

def show_answer_lengths():
    a = variants[(variants.variant_type == "A") & (variants.text != "")].copy()
    if a.empty:
        return
    state = _attempt_state()
    a["words"] = a.text.map(n_words)
    a["attempts"] = [state.get(v, (1, False))[0] for v in a.variant_id]
    a["status"] = [answer_status(w, state.get(v, (1, False))[1]) for w, v in zip(a.words, a.variant_id)]
    print(f"\nОтветы A (допустимо {A_MIN_WORDS}-{A_MAX_WORDS} слов):")
    print(a.groupby("generator_model").agg(n=("words", "size"), median_words=("words", "median"),
                                           regenerated=("attempts", lambda x: int((x > 1).sum()))).to_string())
    print(a.groupby(["generator_model", "status"]).size().unstack(fill_value=0).to_string())


def stage_answers():
    for m in EVALUATED_MODELS:                 # модели строго по очереди
        try:
            run_generation(m, unload=False)    # 1) первичная генерация (естественные ответы)
            regenerate_answers(m)              # 2) перегенерация слишком длинных/коротких/обрезанных
        finally:
            unload_model(m)
            sync_task_answers()




def stage_variants():
    global variants
    variants = ensure_derived_rows(variants, tasks)
    write_csv_atomic(variants, VARIANTS_CSV)
    for m in dict.fromkeys([GENERATOR_MODEL, REWRITER_MODEL]):
        run_generation(m)


# ---------- судьи ----------
def parse_winner(raw):
    m = re.match(r"\s*[\[\(\"'*]*\s*(?:Response\s+|Answer\s+)?([AB])\b", raw)
    if not m:
        return "invalid"
    return "left" if m.group(1) == "A" else "right"

def run_judging(judge_label):
    global judgments
    check_budget()
    model_id = JUDGE_MODELS[judge_label]
    text_of = dict(zip(variants.variant_id, variants.text))
    instr = dict(zip(tasks.task_id, tasks.instruction))
    done = set(judgments.comparison_id)
    c = comparisons[(comparisons.judge_id == judge_label) & ~comparisons.comparison_id.isin(done)
                    & comparisons.task_id.isin(active_task_ids())]
    c = c[(c.variant_left_id.map(text_of).fillna("") != "") & (c.variant_right_id.map(text_of).fillna("") != "")]
    rows = sorted(c.itertuples(), key=lambda r: -(len(text_of[r.variant_left_id]) + len(text_of[r.variant_right_id])))
    if not rows:
        print(f"[skip] {judge_label}: нечего судить"); return
    print(f"[run] {judge_label} ({model_id}): осталось {len(rows)}")
    bar = tqdm(total=len(rows), desc=judge_label)
    try:
        for chunk in chunks(rows, BATCH_JUDGE):
            check_budget()
            prompts = [JUDGE_PROMPT.format(instruction=instr[r.task_id], left=text_of[r.variant_left_id],
                                           right=text_of[r.variant_right_id]) for r in chunk]
            res = generate_safe(model_id, prompts, JUDGE_MAX_NEW_TOKENS)
            new = [{"judgment_id": f"J_{r.comparison_id}", "comparison_id": r.comparison_id,
                    "winner": parse_winner(x["text"]), "raw_response": x["text"],
                    "latency_ms": round(x["latency_ms"], 1), "input_tokens": x["input_tokens"],
                    "output_tokens": x["output_tokens"]} for r, x in zip(chunk, res)]
            judgments = pd.concat([judgments, pd.DataFrame(new, columns=JUDGMENT_COLS)], ignore_index=True)
            write_csv_atomic(judgments, JUDGMENTS_CSV)
            bar.update(len(chunk))
    finally:
        bar.close()
        unload_model(model_id)

def stage_judging():
    for label in JUDGES:           
        run_judging(label)


def safe(stage_fn):
    try:
        stage_fn()
    except BudgetExceeded:
        print("Достигнут лимит времени: всё сохранено. Перезапустите ноутбук и продолжите с этой же ячейки.")

def run_all():
    for fn in (stage_answers, stage_variants, stage_judging):
        safe(fn)


def show_status():
    a = variants[variants.variant_type == "A"]
    print("A-ответы:", int((a.text != "").sum()), "/", len(a))
    d = variants[variants.variant_type != "A"]
    if len(d):
        print("\nВарианты (готово/всего):")
        print(d.assign(done=d.text != "").groupby("variant_type").done.agg(["sum", "count"]).to_string())
        ok = d[d.text != ""].copy()
        if len(ok):
            ok["rel"] = ok.actual_length_ratio.astype(float) / ok.target_length_ratio.astype(float)
            print("\nФактическая/целевая длина (медиана) - должна быть ~1:")
            print(ok.groupby(["variant_type", "target_length_ratio"]).rel.median().round(2).to_string())
    print("\nСудейство (готово/всего):")
    done = set(judgments.comparison_id)
    print(comparisons.assign(done=comparisons.comparison_id.isin(done))
          .groupby("judge_id").done.agg(["sum", "count"]).to_string())
    if len(judgments):
        print("\nНевалидных вердиктов:", int((judgments.winner == "invalid").sum()), "из", len(judgments))


## Stage 1 - ответы оцениваемых моделей (вариант A)

In [10]:
from huggingface_hub import login
login()

In [20]:
# Stage 1: оцениваемые модели отвечают на все вопросы (по очереди, с выгрузкой из cuda)
safe(stage_answers)

[run] HuggingFaceTB/SmolLM2-1.7B-Instruct: осталось 5


SmolLM2-1.7B-Instruct:   0%|          | 0/5 [00:00<?, ?it/s]

[unload] HuggingFaceTB/SmolLM2-1.7B-Instruct | GPU: cuda:0 0.0 GiB, cuda:1 0.0 GiB


NameError: name 'gen_kwargs' is not defined

In [ ]:
df = pd.read_csv('/kaggle/working/data/tasks.csv')

df.head(30)

In [30]:
subset = df.iloc[0:7]
print(subset["answers"])

0    [{"generator": "google/gemma-2-9b-it", "output...
1    [{"generator": "google/gemma-2-9b-it", "output...
2    [{"generator": "google/gemma-2-9b-it", "output...
3    [{"generator": "google/gemma-2-9b-it", "output...
4    [{"generator": "google/gemma-2-9b-it", "output...
5                                                   []
6                                                   []
Name: answers, dtype: object


In [18]:
first_row = df["answers"].iloc[0]
print(first_row)


[{"generator": "google/gemma-2-9b-it", "output": "Don't worry, kickball is a fun and relatively simple game! Here's a breakdown to help you understand the basics and avoid any unfair play:\n\n**The Basics:**\n\n* **Two Teams:** Kickball is played with two teams, just like baseball.\n* **The Field:**  A kickball field is similar to a baseball field, with bases (first, second, third, and home) arranged in a diamond shape.\n* **The Objective:** The goal is to score more runs than the other team. A run is scored when a player successfully kicks the ball and runs around all four bases and back to home plate.\n\n**Gameplay:**\n\n1. **Kicking:** One player from the batting team kicks the ball. They get one kick per turn.\n2. **Running:** After kicking, the kicker runs towards first base. If the ball is caught before it hits the ground, the kicker is out.\n3. **Fielding:** The fielding team tries to catch the ball and get the kicker out. They can also throw the ball to a base to tag a runner o

## Stage 2 - варианты B, C, D, E, P (переписчик)

In [31]:
# Stage 2: переписчик строит варианты B, C, D, E, P
safe(stage_variants)

[run] microsoft/Phi-3.5-mini-instruct: осталось 100


Phi-3.5-mini-instruct:   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

[load] microsoft/Phi-3.5-mini-instruct | GPU: cuda:0 1.0 GiB, cuda:1 1.2 GiB
[unload] microsoft/Phi-3.5-mini-instruct | GPU: cuda:0 1.5 GiB, cuda:1 1.8 GiB


KeyboardInterrupt: 

In [33]:
df = pd.read_csv('/kaggle/working/data/variants.csv')

df.head(30)

,variant_id,task_id,variant_type,text,target_length_ratio,actual_length_ratio,generator_model,prompt
0,T0000_smol_A,T0000,A,"Sure, I'd be happy to explain how to play kick...",1.0,1.0,HuggingFaceTB/SmolLM2-1.7B-Instruct,"Hi, my sister and her girlfriends want me to p..."
1,T0001_smol_A,T0001,A,Wrapping a present neatly is a thoughtful way ...,1.0,1.0,HuggingFaceTB/SmolLM2-1.7B-Instruct,How do I wrap a present neatly?
2,T0002_smol_A,T0002,A,"To dice without slicing your finger, follow th...",1.0,1.0,HuggingFaceTB/SmolLM2-1.7B-Instruct,How do I dice without slicing my finger
3,T0003_smol_A,T0003,A,"Sure, I'd be happy to help you with that! Here...",1.0,1.0,HuggingFaceTB/SmolLM2-1.7B-Instruct,I like to host guests at my home from time to ...
4,T0004_smol_A,T0004,A,"Sure, I'd be happy to help you with that. Here...",1.0,1.0,HuggingFaceTB/SmolLM2-1.7B-Instruct,I like to host guests at my home from time to ...
5,T0005_smol_A,T0005,A,NaN,1.0,NaN,HuggingFaceTB/SmolLM2-1.7B-Instruct,How do you become an author?
6,T0006_smol_A,T0006,A,NaN,1.0,NaN,HuggingFaceTB/SmolLM2-1.7B-Instruct,How do I detail a car?
7,T0007_smol_A,T0007,A,NaN,1.0,NaN,HuggingFaceTB/SmolLM2-1.7B-Instruct,"Hi, I have some falafel, but no tahini to put ..."
8,T0008_smol_A,T0008,A,NaN,1.0,NaN,HuggingFaceTB/SmolLM2-1.7B-Instruct,I like to host guests at my home from time to ...
9,T0009_smol_A,T0009,A,NaN,1.0,NaN,HuggingFaceTB/SmolLM2-1.7B-Instruct,How do I take care of a wooden table?


In [34]:
df.isnull().sum()

variant_id               0
task_id                  0
variant_type             0
text                   494
target_length_ratio      0
actual_length_ratio    494
generator_model          0
prompt                   0
dtype: int64

## Stage 3 - судьи

In [35]:
# Stage 3: судьи выносят вердикты (по очереди)
safe(stage_judging)

[run] judge_qwen (Qwen/Qwen2.5-7B-Instruct): осталось 18


judge_qwen:   0%|          | 0/18 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[load] meta-llama/Llama-3.2-3B-Instruct | GPU: cuda:0 3.2 GiB, cuda:1 2.2 GiB


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[unload] meta-llama/Llama-3.2-3B-Instruct | GPU: cuda:0 1.5 GiB, cuda:1 1.8 GiB


In [36]:
df = pd.read_csv('/kaggle/working/data/judgments.csv')

df.head(30)

,judgment_id,comparison_id,winner,raw_response,latency_ms,input_tokens,output_tokens
0,J_T0002_gemma_exp1_B_vs_C_B200_C200_judge_qwen_r0,T0002_gemma_exp1_B_vs_C_B200_C200_judge_qwen_r0,right,B,1714.1,1760,2
1,J_T0002_gemma_exp1_B_vs_C_B200_C200_judge_qwen_r1,T0002_gemma_exp1_B_vs_C_B200_C200_judge_qwen_r1,left,A,1714.1,1760,2
2,J_T0002_gemma_exp1_B_vs_C_B125_C125_judge_qwen_r0,T0002_gemma_exp1_B_vs_C_B125_C125_judge_qwen_r0,left,A,1714.1,1739,2
3,J_T0002_gemma_exp1_B_vs_C_B125_C125_judge_qwen_r1,T0002_gemma_exp1_B_vs_C_B125_C125_judge_qwen_r1,left,A,1714.1,1739,2
4,J_T0002_gemma_exp1_B_vs_C_B150_C150_judge_qwen_r0,T0002_gemma_exp1_B_vs_C_B150_C150_judge_qwen_r0,right,B,1714.1,1725,2
5,J_T0002_gemma_exp1_B_vs_C_B150_C150_judge_qwen_r1,T0002_gemma_exp1_B_vs_C_B150_C150_judge_qwen_r1,right,B,1714.1,1725,2
6,J_T0002_gemma_exp2_C_vs_A_C200_A_judge_qwen_r0,T0002_gemma_exp2_C_vs_A_C200_A_judge_qwen_r0,left,A,1714.1,1343,2
7,J_T0002_gemma_exp2_C_vs_A_C200_A_judge_qwen_r1,T0002_gemma_exp2_C_vs_A_C200_A_judge_qwen_r1,left,A,1714.1,1343,2
8,J_T0002_gemma_exp2_C_vs_A_C125_A_judge_qwen_r0,T0002_gemma_exp2_C_vs_A_C125_A_judge_qwen_r0,left,A,1121.6,1317,2
9,J_T0002_gemma_exp2_C_vs_A_C125_A_judge_qwen_r1,T0002_gemma_exp2_C_vs_A_C125_A_judge_qwen_r1,left,A,1121.6,1317,2


## Статус и контроль качества

In [38]:
show_status()

A-ответы: 10 / 416

Варианты (готово/всего):
              sum  count
variant_type            
B               6     30
C               3     30
D               2     20
E               1     10
P               0     10

Фактическая/целевая длина (медиана) - должна быть ~1:
variant_type  target_length_ratio
B             1.25                   1.51
              1.50                   1.18
              2.00                   0.98
C             1.25                   1.07
              1.50                   0.85
              2.00                   0.69
D             0.50                   0.59
              0.75                   0.51
E             0.75                   0.58

Судейство (готово/всего):
               sum  count
judge_id                 
judge_llama     18   8320
judge_mistral   18   8320
judge_qwen      18   8320

Невалидных вердиктов: 0 из 54


In [22]:
import os, gc, re, json, time, shutil, hashlib
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

_T0 = time.time()

# ============================== КОНФИГУРАЦИЯ ==============================
SEED = 42
N_TASKS = 250                  # 200-300
ACTIVE_TASK_LIMIT = 5 #None       # None = все задачи; число K = тест на первых K задачах

# Файлы создаются\читаются в директории ноутбука
DATA_DIR = Path.cwd() / "data"
TASKS_CSV       = DATA_DIR / "tasks.csv"
VARIANTS_CSV    = DATA_DIR / "variants.csv"
COMPARISONS_CSV = DATA_DIR / "comparisons.csv"
JUDGMENTS_CSV   = DATA_DIR / "judgments.csv"
ATTEMPTS_CSV    = DATA_DIR / "answer_attempts.csv"   # журнал попыток генерации ответов A

# Модели
EVALUATED_MODELS = [                       # каждая отвечает на все задачи, создается вариант A для каждой задачи моделью
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "google/gemma-2-9b-it",
]
SOURCE_TAGS = {                            # короткие метки в variant_id/comparison_id 
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": "smol",
    "google/gemma-2-9b-it": "gemma",
}
REWRITER_MODEL = "microsoft/Phi-3.5-mini-instruct"    # вариант P
GENERATOR_MODEL = REWRITER_MODEL                      # варианты B, C, D, E
JUDGE_MODELS = {                                      # метка (в comparisons.csv)
    "judge_qwen":    "Qwen/Qwen2.5-7B-Instruct",
    "judge_mistral": "mistralai/Mistral-Nemo-Instruct-2407",
    "judge_llama":   "meta-llama/Llama-3.2-3B-Instruct",
}
JUDGES = list(JUDGE_MODELS)
PRIVATE_MODELS = {"meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-9b-it"}  # нужен HF_TOKEN

# Уровни длины
EXPAND_LEVELS   = [1.25, 1.50, 2.00]       # B и C
COMPRESS_LEVELS = [0.75, 0.50]             # D
LOSSY_LEVELS    = [0.75]                   # E
REPEATS = 2                                # repeat_index 0/1: позиции left/right меняются местами

# Генерация
USE_4BIT = True
BATCH_GEN = 4                              # батч при генерации ответов и вариантов
BATCH_JUDGE = 8                            # батч при судействе
A_MAX_NEW_TOKENS = 700

# Контроль длины ответа A оцениваемой модели (перегенерация)
A_MIN_WORDS, A_MAX_WORDS = 100, 450        # допустимая длина ответа: от неё зависит возможность сжатия до 50% и расширения до 200%
A_MAX_REGEN = 3                            # максимум перегенераций одного ответа
A_TARGET_WORDS = 275                       # ориентир длины в подсказке при перегенерации
JUDGE_MAX_NEW_TOKENS = 8
TIME_BUDGET_HOURS = 11.0                   # мягкая остановка до лимита сессии Kaggle (ч)

# Критерии отбора (по ответу AlpacaEval)
MIN_WORDS, MAX_WORDS = 100, 450
MIN_SENTENCES = 6
MAX_CODE_FRACTION = 0.30
MIN_ASCII_RATIO = 0.95

TASK_COLS = ["task_id", "source_benchmark", "benchmark_id", "instruction",
             "answers", "gold_answer", "category", "selection_reason"]
VARIANT_COLS = ["variant_id", "task_id", "variant_type", "text", "target_length_ratio",
                "actual_length_ratio", "generator_model", "prompt"]
COMPARISON_COLS = ["comparison_id", "task_id", "experiment_type", "variant_left_id",
                   "variant_right_id", "judge_id", "repeat_index", "random_seed"]
JUDGMENT_COLS = ["judgment_id", "comparison_id", "winner", "raw_response",
                 "latency_ms", "input_tokens", "output_tokens"]
ATTEMPT_COLS = ["variant_id", "attempt", "n_words", "n_tokens", "truncated", "status", "kept", "hint"]

DERIVED_PLAN = ([("B", r) for r in EXPAND_LEVELS] + [("C", r) for r in EXPAND_LEVELS]
                + [("D", r) for r in COMPRESS_LEVELS] + [("E", r) for r in LOSSY_LEVELS]
                + [("P", 1.0)])

PROMPTS = {
    "B": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so it is about "
          "{words} words long. Add ONLY new, relevant, factually correct atomic claims "
          "(extra facts, examples, caveats). Keep every original claim. Output only the answer."),
    "C": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer so it is about "
          "{words} words long WITHOUT adding any new facts or claims: only paraphrase, add "
          "explanatory connectives, restate and structure. Output only the answer."),
    "D": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nCondense the answer to about "
          "{words} words while keeping ALL essential atomic claims. Remove only redundancy. "
          "The text must stay fluent. Output only the answer."),
    "E": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nShorten the answer to about "
          "{words} words by DELETING some essential points (about a third of the key claims). "
          "Do not add anything new. Output only the answer."),
    "P": ("Instruction: {instruction}\n\nAnswer:\n{answer}\n\nRewrite the answer in different "
          "wording and style with the SAME length (~{words} words) and exactly the same "
          "claims. Output only the answer."),
}

JUDGE_PROMPT = ("You are an impartial judge. Below are an instruction and two responses to it. "
                "Decide which response is better. Do not let the order of the responses influence "
                "your decision.\n\n[Instruction]\n{instruction}\n\n[Response A]\n{left}\n\n"
                "[Response B]\n{right}\n\nWhich response is better? "
                "Answer with a single letter: A or B.")


In [24]:
# ============================== HF TOKEN ==============================
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("ВНИМАНИЕ: HF_TOKEN не найден - gated-модели (Llama, Gemma) не загрузятся. "
          "Добавьте токен в Kaggle Secrets.")

quantization_config = None
if torch.cuda.is_available() and USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

_loaded_models = {}


def get_hf_auth_kwargs(model_id):
    return {"token": HF_TOKEN} if (model_id in PRIVATE_MODELS and HF_TOKEN) else {}


def gpu_mem():
    if not torch.cuda.is_available():
        return "cpu"
    return ", ".join(f"cuda:{i} {torch.cuda.memory_allocated(i)/2**30:.1f} GiB"
                     for i in range(torch.cuda.device_count()))


def load_local_model(model_id):
    if model_id in _loaded_models:
        return _loaded_models[model_id]
    auth = get_hf_auth_kwargs(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left", **auth)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"device_map": "auto",
              "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32, **auth}
    if quantization_config is not None:
        kwargs["quantization_config"] = quantization_config
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    _loaded_models[model_id] = (tokenizer, model)
    print(f"[load] {model_id} | GPU: {gpu_mem()}")
    return _loaded_models[model_id]


def unload_model(model_id):
    """Полностью освобождает CUDA под следующую модель."""
    entry = _loaded_models.pop(model_id, None)
    if entry is not None:
        tokenizer, model = entry
        del entry, tokenizer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print(f"[unload] {model_id} | GPU: {gpu_mem()}")


@torch.inference_mode()
def generate_batch(model_id, prompts, max_new_tokens, seed=None, **gen_kwargs):
    """Жадная генерация батча. Возвращает список dict(text, input_tokens, output_tokens, latency_ms).
    latency_ms - время батча/размер батча."""
    tokenizer, model = load_local_model(model_id)
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True)
             for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)   # BOS уже есть в chat-шаблоне
    t0 = time.perf_counter()
    if seed is not None:
        torch.manual_seed(seed)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id,
                         **{"do_sample": False, **gen_kwargs})
    elapsed_ms = (time.perf_counter() - t0) * 1000
    in_len = enc["input_ids"].shape[1]
    in_tokens = enc["attention_mask"].sum(dim=1).tolist()
    res = []
    for i in range(len(prompts)):
        new = out[i, in_len:]
        res.append({"text": tokenizer.decode(new, skip_special_tokens=True).strip(),
                    "input_tokens": int(in_tokens[i]),
                    "output_tokens": int((new != tokenizer.pad_token_id).sum()),
                    "latency_ms": elapsed_ms / len(prompts)})
    return res


def generate_safe(model_id, prompts, max_new_tokens, **gen_kwargs):
    """При нехватке памяти батч делится пополам."""
    try:
        return generate_batch(model_id, prompts, max_new_tokens, **gen_kwargs)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(prompts) == 1:
            raise
        mid = len(prompts) // 2
        return (generate_safe(model_id, prompts[:mid], max_new_tokens, **gen_kwargs)
                + generate_safe(model_id, prompts[mid:], max_new_tokens, **gen_kwargs))



# ============================== Вспомогательные функции ==============================
def n_words(text): return len(text.split())
def n_sentences(text): return len([s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.split()) >= 3])
def code_fraction(text):
    code = sum(len(m) for m in re.findall(r"```.*?```", text, flags=re.S))
    return code / max(len(text), 1)
def ascii_ratio(text): return sum(c.isascii() for c in text) / max(len(text), 1)
def stable_int(*parts, mod=2**31 - 1):
    return int(hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest(), 16) % mod
def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def variant_id(task_id, src, vt, ratio=1.0):
    """T0001_smol_A, T0001_smol_B125, ... src - метка оцениваемой модели, чей ответ A расширяется/сжимается."""
    return f"{task_id}_{src}_A" if vt == "A" else f"{task_id}_{src}_{vt}{round(ratio * 100)}"

def a_id_of(vid):
    t, s = vid.split("_")[:2]
    return f"{t}_{s}_A"

# ============================== CSV вывод и запись ==============================
def read_csv(path, cols):
    df = pd.read_csv(path, dtype=object, keep_default_na=False)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: нет колонок {missing}")
    return df[cols].copy()

def write_csv_atomic(df, path):
    """Запись через временный файл"""
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        df.to_csv(f, index=False)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# ============================== 1. tasks ==============================
def load_alpaca_eval():
    url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
    df = pd.read_json(url).reset_index(drop=True)      
    df.insert(0, "benchmark_id", range(len(df)))
    return df                 # benchmark_id, instruction, output, generator, dataset

def select_tasks(df, n_tasks):
    df = df.copy()
    df["words"] = df["output"].map(n_words)
    df["sents"] = df["output"].map(n_sentences)
    df["code_frac"] = df["output"].map(code_fraction)
    df["ascii"] = (df["instruction"] + df["output"]).map(ascii_ratio)
    ok = df[(df.words.between(MIN_WORDS, MAX_WORDS)) & (df.sents >= MIN_SENTENCES)
            & (df.code_frac <= MAX_CODE_FRACTION) & (df.ascii >= MIN_ASCII_RATIO)].copy()
    ok = ok.drop_duplicates("instruction")
    if len(ok) < 200:
        raise RuntimeError(f"После фильтрации осталось {len(ok)} задач (<200): ослабьте критерии.")
    n_tasks = min(n_tasks, len(ok))
    shares = ok["dataset"].value_counts(normalize=True)          # стратификация по подмножествам
    parts = []
    for cat, share in shares.items():
        sub = ok[ok.dataset == cat]
        parts.append(sub.sample(n=min(max(1, round(share * n_tasks)), len(sub)), random_state=SEED))
    sel = pd.concat(parts)
    if len(sel) > n_tasks:
        sel = sel.sample(n=n_tasks, random_state=SEED)
    elif len(sel) < n_tasks:
        sel = pd.concat([sel, ok.drop(sel.index).sample(n=n_tasks - len(sel), random_state=SEED)])
    sel = sel.sort_values("benchmark_id").reset_index(drop=True)
    rows = []
    for i, r in sel.iterrows():
        rows.append({
            "task_id": f"T{i:04d}", "source_benchmark": "AlpacaEval 2.0",
            "benchmark_id": int(r.benchmark_id), "instruction": r.instruction,
            "answers": "[]",                       # заполняется после Stage 1 (ответ оцениваемой модели)
            "gold_answer": r.output,               # референс AlpacaEval (GPT-4 Turbo), не строгий gold
            "category": r.dataset,
            "selection_reason": (f"reference {r.words} words, {r.sents} sentences, "
                                 f"code_frac={r.code_frac:.2f}: enough content for 50-75% compression "
                                 f"and 125-200% expansion; stratified by subset '{r.dataset}'")})
    return pd.DataFrame(rows, columns=TASK_COLS)


# ============================== 2. variants ==============================
def ensure_a_rows(variants, tasks):
    """Строки A для всех задач; каждая модель отвечает на каждую задачу."""
    have = set(variants.variant_id)
    new = []
    '''
    for i, t in enumerate(tasks.itertuples()):
        vid = variant_id(t.task_id, "A")
        if vid not in have:
            new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                        "target_length_ratio": 1.0, "actual_length_ratio": "",
                        #"generator_model": EVALUATED_MODELS[i % len(EVALUATED_MODELS)],
                        "generator_model": EVALUATED_MODELS[i],
                        "prompt": t.instruction})
    '''
    for model in EVALUATED_MODELS:
        for t in tasks.itertuples():
            vid = variant_id(t.task_id, SOURCE_TAGS[model], "A")
            if vid not in have:
                new.append({"variant_id": vid, "task_id": t.task_id, "variant_type": "A", "text": "",
                            "target_length_ratio": 1.0, "actual_length_ratio": "",
                            "generator_model": model, "prompt": t.instruction})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants

def ensure_derived_rows(variants, tasks):
    """Строки B/C/D/E/P с готовым prompt - только для задач, у которых уже есть текст A."""
    have = set(variants.variant_id)
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    instr = dict(zip(tasks.task_id, tasks.instruction))
    new = []
    for r in a.itertuples():
        src = SOURCE_TAGS[r.generator_model]
        a_words = n_words(r.text)
        for vt, ratio in DERIVED_PLAN:
            vid = variant_id(r.task_id, src, vt, ratio)
            if vid in have:
                continue
            prompt = PROMPTS[vt].format(instruction=instr[r.task_id], answer=r.text,
                                        words=round(a_words * ratio))
            new.append({"variant_id": vid, "task_id": r.task_id, "variant_type": vt, "text": "",
                        "target_length_ratio": ratio, "actual_length_ratio": "",
                        "generator_model": REWRITER_MODEL if vt == "P" else GENERATOR_MODEL,
                        "prompt": prompt})
    if new:
        variants = pd.concat([variants, pd.DataFrame(new, columns=VARIANT_COLS)], ignore_index=True)
    return variants


# ============================== 3. comparisons ==============================
def build_comparisons(task_ids):
    rows = []
    def add(task_id, src, exp, v1, v2):
        for judge in JUDGES:
            for rep in range(REPEATS):
                left, right = (v1, v2) if rep % 2 == 0 else (v2, v1)
                cid = f"{task_id}_{src}_{exp}_{v1.split('_')[2]}_{v2.split('_')[2]}_{judge}_r{rep}"
                rows.append({"comparison_id": cid, "task_id": task_id, "experiment_type": exp,
                             "variant_left_id": left, "variant_right_id": right,
                             "judge_id": judge, "repeat_index": rep,
                             "random_seed": stable_int(cid, SEED)})
    for tid in task_ids:
        for model in EVALUATED_MODELS:                 # сравнения только внутри одного источника A
            src = SOURCE_TAGS[model]
            V = lambda vt, r=1.0: variant_id(tid, src, vt, r)
            for r in EXPAND_LEVELS:
                add(tid, src, "exp1_B_vs_C", V("B", r), V("C", r))
                add(tid, src, "exp2_C_vs_A", V("C", r), V("A"))
            for r in COMPRESS_LEVELS:
                add(tid, src, "exp3_D_vs_A", V("D", r), V("A"))
            for r in LOSSY_LEVELS:
                add(tid, src, "exp4_E_vs_A", V("E", r), V("A"))
            add(tid, src, "exp5_P_vs_A", V("P"), V("A"))
    return pd.DataFrame(rows, columns=COMPARISON_COLS)





# =============== Init: файлы создаются, если их нет; если есть - загрузка и продолжение ===============
DATA_DIR.mkdir(parents=True, exist_ok=True)

def restore_from_previous_run():
    """Если в папке ноутбука данных нет, а результаты прошлого запуска подключены как Kaggle Input — копируем их."""
    if TASKS_CSV.exists():
        return
    root = Path("/kaggle/input")
    if not root.exists():
        return
    names = ["tasks.csv", "variants.csv", "comparisons.csv", "judgments.csv"]
    for p in root.rglob("tasks.csv"):
        if all((p.parent / n).exists() for n in names):
            for n in names + ["answer_attempts.csv"]:
                if (p.parent / n).exists():
                    shutil.copy(p.parent / n, DATA_DIR / n)
            print(f"[restore] данные восстановлены из {p.parent}")
            return

restore_from_previous_run()

# tasks
if TASKS_CSV.exists():
    tasks = read_csv(TASKS_CSV, TASK_COLS); print(f"[tasks] загружен: {len(tasks)} задач")
else:
    tasks = select_tasks(load_alpaca_eval(), N_TASKS)
    write_csv_atomic(tasks, TASKS_CSV); print(f"[tasks] создан: {len(tasks)} задач")

# variants (строки A)
if VARIANTS_CSV.exists():
    variants = read_csv(VARIANTS_CSV, VARIANT_COLS); print(f"[variants] загружен: {len(variants)} строк")
else:
    variants = pd.DataFrame(columns=VARIANT_COLS, dtype=object); print("[variants] создан")
variants = ensure_a_rows(variants, tasks)
write_csv_atomic(variants, VARIANTS_CSV)

# comparisons (достраивание недостающих id)
plan = build_comparisons(tasks.task_id)
if COMPARISONS_CSV.exists():
    comparisons = read_csv(COMPARISONS_CSV, COMPARISON_COLS)
    comparisons = pd.concat([comparisons, plan[~plan.comparison_id.isin(set(comparisons.comparison_id))]],
                            ignore_index=True)
    print(f"[comparisons] загружен: {len(comparisons)} строк")
else:
    comparisons = plan; print(f"[comparisons] создан: {len(comparisons)} строк")
write_csv_atomic(comparisons, COMPARISONS_CSV)

# judgments
if JUDGMENTS_CSV.exists():
    judgments = read_csv(JUDGMENTS_CSV, JUDGMENT_COLS); print(f"[judgments] загружен: {len(judgments)} строк")
else:
    judgments = pd.DataFrame(columns=JUDGMENT_COLS, dtype=object)
    write_csv_atomic(judgments, JUDGMENTS_CSV); print("[judgments] создан (пустой)")

# журнал попыток генерации ответов A
if ATTEMPTS_CSV.exists():
    attempts = read_csv(ATTEMPTS_CSV, ATTEMPT_COLS); print(f"[attempts] загружен: {len(attempts)} строк")
else:
    attempts = pd.DataFrame(columns=ATTEMPT_COLS, dtype=object)
    write_csv_atomic(attempts, ATTEMPTS_CSV); print("[attempts] создан (пустой)")


[tasks] загружен: 208 задач
[variants] загружен: 416 строк
[comparisons] загружен: 24960 строк
[judgments] загружен: 0 строк
[attempts] загружен: 0 строк


In [ ]:
class BudgetExceeded(Exception):
    pass

def check_budget():
    if time.time() - _T0 > TIME_BUDGET_HOURS * 3600:
        raise BudgetExceeded

def active_task_ids():
    ids = tasks.task_id.tolist()
    return set(ids[:ACTIVE_TASK_LIMIT]) if ACTIVE_TASK_LIMIT else set(ids)

def sync_task_answers():
    """Переносит ответы A оцениваемых моделей в tasks.answers."""
    global tasks
    a = variants[(variants.variant_type == "A") & (variants.text != "")]
    ans = {}
    for r in a.itertuples():
        ans.setdefault(r.task_id, []).append({"generator": r.generator_model, "output": r.text})
    tasks["answers"] = [json.dumps(ans[t], ensure_ascii=False) if t in ans else old
                        for t, old in zip(tasks.task_id, tasks.answers)]
    write_csv_atomic(tasks, TASKS_CSV)


# ---------- генерация (ответы A и варианты B/C/D/E/P) ----------
def run_generation(model_id, unload=True):
    """Дозаполняет все пустые тексты, закреплённые за model_id. После каждого батча сохранение."""
    global variants
    check_budget()
    mask = ((variants.generator_model == model_id) & (variants.text == "")
            & (variants.prompt != "") & variants.task_id.isin(active_task_ids()))
    idx = variants.index[mask].tolist()
    if not idx:
        print(f"[skip] {model_id}: нечего генерировать"); return
    a_words = {r.variant_id: n_words(r.text) for r in variants[variants.variant_type == "A"].itertuples() if r.text}

    def max_new(i):
        row = variants.loc[i]
        if row.variant_type == "A":
            return A_MAX_NEW_TOKENS
        words = round(a_words[a_id_of(row.variant_id)] * float(row.target_length_ratio))
        return min(int(words * 1.5) + 60, 1800)

    idx.sort(key=lambda i: -len(variants.at[i, "prompt"]))     # похожие по длине - в один батч
    print(f"[run] {model_id}: осталось {len(idx)}")
    bar = tqdm(total=len(idx), desc=model_id.split("/")[-1])
    try:
        for chunk in chunks(idx, BATCH_GEN):
            check_budget()
            res = generate_safe(model_id, [variants.at[i, "prompt"] for i in chunk],
                                max(max_new(i) for i in chunk))
            new_log = []
            for i, r in zip(chunk, res):
                if not r["text"]:
                    continue                                   # пустой вывод - повторится при следующем запуске
                variants.at[i, "text"] = r["text"]
                if variants.at[i, "variant_type"] == "A":
                    variants.at[i, "actual_length_ratio"] = 1.0
                    new_log.append(_attempt_row(variants.at[i, "variant_id"], 1, r, "", True))
                else:
                    variants.at[i, "actual_length_ratio"] = round(
                        n_words(r["text"]) / a_words[a_id_of(variants.at[i, "variant_id"])], 3)
            write_csv_atomic(variants, VARIANTS_CSV)
            if new_log:
                log_attempts(new_log)
            bar.update(len(chunk))
    finally:
        bar.close()
        if unload:
            unload_model(model_id)
        sync_task_answers()


# ---------- контроль длины ответов A: журнал попыток и перегенерация ----------
import itertools

def answer_status(n_w, truncated=False):
    if truncated:
        return "truncated"
    if n_w > A_MAX_WORDS:
        return "too_long"
    if n_w < A_MIN_WORDS:
        return "too_short"
    return "ok"

def badness(n_w, truncated=False):
    """Насколько ответ вне допустимого диапазона (0 - в диапазоне); обрезанный ответ штрафуется."""
    return max(A_MIN_WORDS - n_w, n_w - A_MAX_WORDS, 0) + (1000 if truncated else 0)

def _attempt_row(vid, attempt, res, hint, kept):
    n_w = n_words(res["text"])
    trunc = int(res["output_tokens"] >= A_MAX_NEW_TOKENS)       # упёрлись в лимит токенов -> ответ обрезан
    return {"variant_id": vid, "attempt": attempt, "n_words": n_w, "n_tokens": res["output_tokens"],
            "truncated": trunc, "status": answer_status(n_w, bool(trunc)), "kept": int(kept), "hint": hint}

def log_attempts(rows):
    global attempts
    attempts = pd.concat([attempts, pd.DataFrame(rows, columns=ATTEMPT_COLS)], ignore_index=True)
    write_csv_atomic(attempts, ATTEMPTS_CSV)

def _attempt_state():
    """{variant_id: (число попыток, обрезан ли сохранённый ответ)}; нет записи в журнале = 1 попытка."""
    if attempts.empty:
        return {}
    a = attempts.copy()
    for c in ("attempt", "truncated", "kept"):
        a[c] = pd.to_numeric(a[c])
    state = {}
    for vid, g in a.groupby("variant_id"):
        kept = g[g.kept == 1]
        state[vid] = (int(g.attempt.max()), bool(kept.truncated.iloc[-1]) if len(kept) else False)
    return state

def answers_needing_regen(model_id):
    """Ответы A модели вне допустимой длины (или обрезанные), у которых остались попытки."""
    state = _attempt_state()
    has_derived = {a_id_of(v) for v in variants.variant_id[variants.variant_type != "A"]}
    a = variants[(variants.variant_type == "A") & (variants.generator_model == model_id)
                 & (variants.text != "") & variants.task_id.isin(active_task_ids())]
    todo, blocked = [], 0
    for i, r in a.iterrows():
        used, trunc = state.get(r.variant_id, (1, False))
        status = answer_status(n_words(r.text), trunc)
        if status == "ok" or used >= 1 + A_MAX_REGEN:
            continue
        if r.variant_id in has_derived:
            blocked += 1
            continue
        todo.append({"idx": i, "vid": r.variant_id, "task_id": r.task_id, "status": status,
                     "attempt": used + 1, "old_bad": badness(n_words(r.text), trunc)})
    if blocked:
        print(f"[regen] пропущено {blocked}: от этих ответов уже построены B-E/P (см. reset_answers)")
    return todo

def regen_prompt(instruction, status):
    if status == "too_short":
        hint = f"Please give a detailed and complete answer of about {A_TARGET_WORDS} words."
    else:                                            # too_long / truncated
        hint = f"Please answer concisely, in about {A_TARGET_WORDS} words."
    return f"{instruction}\n\n{hint}", hint

def regenerate_answers(model_id):
    """Перегенерирует слишком длинные/короткие/обрезанные ответы A (модель остаётся загруженной).
    Попытка 2: жадная генерация + подсказка о длине; попытки 3+: сэмплирование (T=0.7) + подсказка.
    Новый ответ заменяет старый, только если ближе к допустимому диапазону. Использованный промпт пишется в variants.prompt."""
    instr = dict(zip(tasks.task_id, tasks.instruction))
    for _ in range(A_MAX_REGEN):
        todo = answers_needing_regen(model_id)
        if not todo:
            return
        print(f"[regen] {model_id}: {len(todo)} ответов вне [{A_MIN_WORDS}, {A_MAX_WORDS}] слов или обрезаны")
        todo.sort(key=lambda t: t["attempt"])
        for attempt_no, grp in itertools.groupby(todo, key=lambda t: t["attempt"]):
            kw = {} if attempt_no == 2 else {"do_sample": True, "temperature": 0.7, "top_p": 0.9,
                                              "seed": SEED + attempt_no}
            for chunk in chunks(list(grp), BATCH_GEN):
                check_budget()
                built = [regen_prompt(instr[t["task_id"]], t["status"]) for t in chunk]
                res = generate_safe(model_id, [b[0] for b in built], A_MAX_NEW_TOKENS, **kw)
                log = []
                for t, (prompt, hint), r in zip(chunk, built, res):
                    new_bad = (badness(n_words(r["text"]), r["output_tokens"] >= A_MAX_NEW_TOKENS)
                               if r["text"] else float("inf"))
                    keep_new = new_bad < t["old_bad"]
                    if keep_new:
                        variants.at[t["idx"], "text"] = r["text"]
                        variants.at[t["idx"], "prompt"] = prompt
                    log.append(_attempt_row(t["vid"], attempt_no, r, hint, keep_new))
                write_csv_atomic(variants, VARIANTS_CSV)
                log_attempts(log)

def reset_answers(task_ids, model_id=None):
    """Стирает ответы A выбранных задач (по одной или всем оцениваемым моделям) и всё, что из них построено:
    варианты B-E/P, вердикты судей, журнал попыток. После этого Stage 1 сгенерирует эти ответы заново."""
    global variants, judgments, attempts
    tags = [SOURCE_TAGS[m] for m in ([model_id] if model_id else EVALUATED_MODELS)]
    prefixes = tuple(f"{t}_{s}_" for t in task_ids for s in tags)
    chain = variants.variant_id.str.startswith(prefixes)
    chain_ids = set(variants.variant_id[chain])
    instr = dict(zip(tasks.task_id, tasks.instruction))
    comp_ids = set(comparisons.comparison_id[comparisons.variant_left_id.isin(chain_ids)
                                             | comparisons.variant_right_id.isin(chain_ids)])
    variants = variants[~(chain & (variants.variant_type != "A"))].reset_index(drop=True)
    is_a = variants.variant_id.isin(chain_ids) & (variants.variant_type == "A")
    variants.loc[is_a, ["text", "actual_length_ratio"]] = ""
    variants.loc[is_a, "prompt"] = variants.loc[is_a, "task_id"].map(instr)
    judgments = judgments[~judgments.comparison_id.isin(comp_ids)].reset_index(drop=True)
    attempts = attempts[~attempts.variant_id.isin(chain_ids)].reset_index(drop=True)
    for df, path in ((variants, VARIANTS_CSV), (judgments, JUDGMENTS_CSV), (attempts, ATTEMPTS_CSV)):
        write_csv_atomic(df, path)
    sync_task_answers()

def show_answer_lengths():
    a = variants[(variants.variant_type == "A") & (variants.text != "")].copy()
    if a.empty:
        return
    state = _attempt_state()
    a["words"] = a.text.map(n_words)
    a["attempts"] = [state.get(v, (1, False))[0] for v in a.variant_id]
    a["status"] = [answer_status(w, state.get(v, (1, False))[1]) for w, v in zip(a.words, a.variant_id)]
    print(f"\nОтветы A (допустимо {A_MIN_WORDS}-{A_MAX_WORDS} слов):")
    print(a.groupby("generator_model").agg(n=("words", "size"), median_words=("words", "median"),
                                           regenerated=("attempts", lambda x: int((x > 1).sum()))).to_string())
    print(a.groupby(["generator_model", "status"]).size().unstack(fill_value=0).to_string())


def stage_answers():
    for m in EVALUATED_MODELS:                 # модели строго по очереди
        try:
            run_generation(m, unload=False)    # 1) первичная генерация (естественные ответы)
            regenerate_answers(m)              # 2) перегенерация слишком длинных/коротких/обрезанных
        finally:
            unload_model(m)
            sync_task_answers()

def stage_variants():
    global variants
    variants = ensure_derived_rows(variants, tasks)
    write_csv_atomic(variants, VARIANTS_CSV)
    for m in dict.fromkeys([GENERATOR_MODEL, REWRITER_MODEL]):
        run_generation(m)


# ---------- судьи ----------
def parse_winner(raw):
    m = re.match(r"\s*[\[\(\"'*]*\s*(?:Response\s+|Answer\s+)?([AB])\b", raw)
    if not m:
        return "invalid"
    return "left" if m.group(1) == "A" else "right"

def run_judging(judge_label):
    global judgments
    check_budget()
    model_id = JUDGE_MODELS[judge_label]
    text_of = dict(zip(variants.variant_id, variants.text))
    instr = dict(zip(tasks.task_id, tasks.instruction))
    done = set(judgments.comparison_id)
    c = comparisons[(comparisons.judge_id == judge_label) & ~comparisons.comparison_id.isin(done)
                    & comparisons.task_id.isin(active_task_ids())]
    c = c[(c.variant_left_id.map(text_of).fillna("") != "") & (c.variant_right_id.map(text_of).fillna("") != "")]
    rows = sorted(c.itertuples(), key=lambda r: -(len(text_of[r.variant_left_id]) + len(text_of[r.variant_right_id])))
    if not rows:
        print(f"[skip] {judge_label}: нечего судить"); return
    print(f"[run] {judge_label} ({model_id}): осталось {len(rows)}")
    bar = tqdm(total=len(rows), desc=judge_label)
    try:
        for chunk in chunks(rows, BATCH_JUDGE):
            check_budget()
            prompts = [JUDGE_PROMPT.format(instruction=instr[r.task_id], left=text_of[r.variant_left_id],
                                           right=text_of[r.variant_right_id]) for r in chunk]
            res = generate_safe(model_id, prompts, JUDGE_MAX_NEW_TOKENS)
            new = [{"judgment_id": f"J_{r.comparison_id}", "comparison_id": r.comparison_id,
                    "winner": parse_winner(x["text"]), "raw_response": x["text"],
                    "latency_ms": round(x["latency_ms"], 1), "input_tokens": x["input_tokens"],
                    "output_tokens": x["output_tokens"]} for r, x in zip(chunk, res)]
            judgments = pd.concat([judgments, pd.DataFrame(new, columns=JUDGMENT_COLS)], ignore_index=True)
            write_csv_atomic(judgments, JUDGMENTS_CSV)
            bar.update(len(chunk))
    finally:
        bar.close()
        unload_model(model_id)

def stage_judging():
    for label in JUDGES:           
        run_judging(label)


def safe(stage_fn):
    try:
        stage_fn()
    except BudgetExceeded:
        print("Достигнут лимит времени: всё сохранено. Перезапустите ноутбук и продолжите с этой же ячейки.")

def run_all():
    for fn in (stage_answers, stage_variants, stage_judging):
        safe(fn)


def show_status():
    a = variants[variants.variant_type == "A"]
    print("A-ответы:", int((a.text != "").sum()), "/", len(a))
    d = variants[variants.variant_type != "A"]
    if len(d):
        print("\nВарианты (готово/всего):")
        print(d.assign(done=d.text != "").groupby("variant_type").done.agg(["sum", "count"]).to_string())
        ok = d[d.text != ""].copy()
        if len(ok):
            ok["rel"] = ok.actual_length_ratio.astype(float) / ok.target_length_ratio.astype(float)
            print("\nФактическая/целевая длина (медиана) - должна быть ~1:")
            print(ok.groupby(["variant_type", "target_length_ratio"]).rel.median().round(2).to_string())
    print("\nСудейство (готово/всего):")
    done = set(judgments.comparison_id)
    print(comparisons.assign(done=comparisons.comparison_id.isin(done))
          .groupby("judge_id").done.agg(["sum", "count"]).to_string())
    if len(judgments):
        print("\nНевалидных вердиктов:", int((judgments.winner == "invalid").sum()), "из", len(judgments))
    show_answer_lengths()


In [25]:
# Stage 1: оцениваемые модели отвечают на все вопросы (по очереди, с выгрузкой из cuda)
safe(stage_answers)

[run] HuggingFaceTB/SmolLM2-1.7B-Instruct: осталось 5


SmolLM2-1.7B-Instruct:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

[load] google/gemma-2-9b-it | GPU: cuda:0 4.4 GiB, cuda:1 1.3 GiB
[unload] google/gemma-2-9b-it | GPU: cuda:0 0.0 GiB, cuda:1 0.0 GiB


In [26]:
# Stage 2: переписчик строит варианты B, C, D, E, P
safe(stage_variants)

[run] microsoft/Phi-3.5-mini-instruct: осталось 100


Phi-3.5-mini-instruct:   0%|          | 0/100 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

[load] microsoft/Phi-3.5-mini-instruct | GPU: cuda:0 1.0 GiB, cuda:1 1.2 GiB
[unload] microsoft/Phi-3.5-mini-instruct | GPU: cuda:0 0.0 GiB, cuda:1 0.0 GiB


In [27]:
# Stage 3: судьи выносят вердикты (по очереди)
safe(stage_judging)

[run] judge_qwen (Qwen/Qwen2.5-7B-Instruct): осталось 200


judge_qwen:   0%|          | 0/200 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[load] meta-llama/Llama-3.2-3B-Instruct | GPU: cuda:0 1.7 GiB, cuda:1 0.4 GiB


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[unload] meta-llama/Llama-3.2-3B-Instruct | GPU: cuda:0 0.0 GiB, cuda:1 0.0 GiB


In [28]:
show_status()

A-ответы: 10 / 416

Варианты (готово/всего):
              sum  count
variant_type            
B              30     30
C              30     30
D              20     20
E              10     10
P              10     10

Фактическая/целевая длина (медиана) - должна быть ~1:
variant_type  target_length_ratio
B             1.25                   1.40
              1.50                   1.20
              2.00                   0.97
C             1.25                   1.46
              1.50                   1.16
              2.00                   0.93
D             0.50                   0.79
              0.75                   0.60
E             0.75                   0.63
P             1.00                   1.03

Судейство (готово/всего):
               sum  count
judge_id                 
judge_llama    200   8320
judge_mistral  200   8320
judge_qwen     200   8320

Невалидных вердиктов: 2 из 600
